# Visual Validation: Firescar Predictions

Qualitative assessment of model predictions on validation samples. Shows RGB imagery, ground truth masks, predicted masks, and error maps side-by-side.

In [ ]:
import os
import sys

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.abspath(".."))
from firescars.dataset import FirescarDataset
from firescars.evaluate import compute_metrics
from firescars.model import FirescarModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_PATH = "/tmp/firescar_chips"
CKPT_PATH = "/tmp/firescar_checkpoints/model_best.pth"
print(f"Device: {device}")

## 1. Load Model and Data

In [ ]:
model = FirescarModel(
    encoder_name="vit_base_patch16_224",
    in_chans=6,
    img_size=224,
    pretrained_encoder=False,
).to(device)

ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.eval()

val_ds = FirescarDataset(DATA_PATH, split="val", augment=False)
print(f"Loaded model (epoch {ckpt['epoch'] + 1}), {len(val_ds)} val samples")

## 2. Prediction Grid

For each sample: RGB | False Colour (SWIR/NIR/R) | Ground Truth | Prediction | Error Map

In [ ]:
# Error map colormap: green=correct, red=false positive, blue=false negative
error_cmap = ListedColormap(["black", "green", "red", "blue"])
error_labels = [
    "TN (unburned correct)",
    "TP (burned correct)",
    "FP (false alarm)",
    "FN (missed burn)",
]

n_samples = min(8, len(val_ds))
indices = np.random.choice(len(val_ds), n_samples, replace=False)

fig, axes = plt.subplots(n_samples, 5, figsize=(20, 4 * n_samples))
if n_samples == 1:
    axes = axes[np.newaxis, :]

for row, idx in enumerate(indices):
    img, mask = val_ds[idx]

    # Predict
    with torch.no_grad():
        logits = model(img.unsqueeze(0).to(device))
        prob = torch.sigmoid(logits).cpu().squeeze().numpy()

    pred = (prob > 0.5).astype(np.uint8)
    gt = mask.numpy()
    img_np = img.numpy()  # (6, 224, 224) normalised [0,1]

    # Compute per-sample IoU
    m = compute_metrics(logits.cpu(), mask.unsqueeze(0))

    # RGB (bands 2,1,0 = red, green, blue)
    rgb = np.clip(img_np[[2, 1, 0]].transpose(1, 2, 0) * 2.5, 0, 1)
    # False colour (SWIR1, NIR, Red)
    fc = np.clip(img_np[[4, 3, 2]].transpose(1, 2, 0) * 2.5, 0, 1)

    # Error map: 0=TN, 1=TP, 2=FP, 3=FN
    error = np.zeros_like(pred, dtype=np.uint8)
    error[(pred == 1) & (gt == 1)] = 1  # TP
    error[(pred == 1) & (gt == 0)] = 2  # FP
    error[(pred == 0) & (gt == 1)] = 3  # FN

    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title("RGB")

    axes[row, 1].imshow(fc)
    axes[row, 1].set_title("False Colour (SWIR/NIR/R)")

    axes[row, 2].imshow(gt, cmap="Reds", vmin=0, vmax=1)
    axes[row, 2].set_title(f"Ground Truth ({gt.mean():.1%} burned)")

    axes[row, 3].imshow(prob, cmap="hot", vmin=0, vmax=1)
    axes[row, 3].set_title(f"Prediction (IoU={m['iou']:.3f})")

    axes[row, 4].imshow(error, cmap=error_cmap, vmin=0, vmax=3)
    axes[row, 4].set_title("Error Map")

for ax in axes.flat:
    ax.axis("off")

# Legend for error map
legend_patches = [
    mpatches.Patch(color="black", label="TN"),
    mpatches.Patch(color="green", label="TP"),
    mpatches.Patch(color="red", label="FP"),
    mpatches.Patch(color="blue", label="FN"),
]
fig.legend(
    handles=legend_patches, loc="lower center", ncol=4, fontsize=11, bbox_to_anchor=(0.5, -0.02)
)

plt.suptitle(
    "Validation Predictions: RGB | False Colour | Ground Truth | Probability | Error",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig("validation_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Probability Histogram

In [ ]:
# Collect all predictions on val set
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)
all_probs_val = []
all_targets_val = []

with torch.no_grad():
    for imgs, masks in val_loader:
        logits = model(imgs.to(device))
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs_val.append(probs.flatten())
        all_targets_val.append(masks.numpy().flatten())

all_probs_val = np.concatenate(all_probs_val)
all_targets_val = np.concatenate(all_targets_val)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(
    all_probs_val[all_targets_val == 0],
    bins=50,
    alpha=0.6,
    color="blue",
    label="Unburned pixels",
    density=True,
)
ax.hist(
    all_probs_val[all_targets_val == 1],
    bins=50,
    alpha=0.6,
    color="red",
    label="Burned pixels",
    density=True,
)
ax.axvline(x=0.5, color="k", linestyle="--", label="Threshold (0.5)")
ax.set_xlabel("Predicted Probability")
ax.set_ylabel("Density")
ax.set_title("Prediction Probability Distribution by Class")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("probability_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Best and Worst Predictions

In [ ]:
# Compute per-sample IoU for all val samples
sample_ious = []
for idx in range(len(val_ds)):
    img, mask = val_ds[idx]
    with torch.no_grad():
        logits = model(img.unsqueeze(0).to(device))
    m = compute_metrics(logits.cpu(), mask.unsqueeze(0))
    sample_ious.append(m["iou"])

sorted_idx = np.argsort(sample_ious)
worst_3 = sorted_idx[:3]
best_3 = sorted_idx[-3:]


def plot_samples(indices, title):
    fig, axes = plt.subplots(len(indices), 4, figsize=(16, 4 * len(indices)))
    if len(indices) == 1:
        axes = axes[np.newaxis, :]
    for row, idx in enumerate(indices):
        img, mask = val_ds[idx]
        with torch.no_grad():
            logits = model(img.unsqueeze(0).to(device))
            prob = torch.sigmoid(logits).cpu().squeeze().numpy()
        img_np = img.numpy()
        rgb = np.clip(img_np[[2, 1, 0]].transpose(1, 2, 0) * 2.5, 0, 1)
        gt = mask.numpy()
        pred = (prob > 0.5).astype(np.float32)

        axes[row, 0].imshow(rgb)
        axes[row, 0].set_title(f"RGB (sample {idx})")
        axes[row, 1].imshow(gt, cmap="Reds", vmin=0, vmax=1)
        axes[row, 1].set_title("Ground Truth")
        axes[row, 2].imshow(prob, cmap="hot", vmin=0, vmax=1)
        axes[row, 2].set_title(f"Probability (IoU={sample_ious[idx]:.3f})")
        # Overlay: green=TP, red=FP, blue=FN
        overlay = np.zeros((*gt.shape, 3))
        overlay[(pred == 1) & (gt == 1)] = [0, 1, 0]  # TP green
        overlay[(pred == 1) & (gt == 0)] = [1, 0, 0]  # FP red
        overlay[(pred == 0) & (gt == 1)] = [0, 0, 1]  # FN blue
        axes[row, 3].imshow(rgb * 0.4 + overlay * 0.6)
        axes[row, 3].set_title("Overlay (G=TP, R=FP, B=FN)")
    for ax in axes.flat:
        ax.axis("off")
    plt.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


plot_samples(best_3, "Best Predictions (Highest IoU)")
plot_samples(worst_3, "Worst Predictions (Lowest IoU)")

## 5. Burn Boundary Quality

In [ ]:
from scipy import ndimage

# Show prediction boundaries overlaid on imagery
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
sample_indices = np.random.choice(len(val_ds), 8, replace=False)

for ax, idx in zip(axes.flat, sample_indices):
    img, mask = val_ds[idx]
    with torch.no_grad():
        logits = model(img.unsqueeze(0).to(device))
        prob = torch.sigmoid(logits).cpu().squeeze().numpy()

    img_np = img.numpy()
    rgb = np.clip(img_np[[2, 1, 0]].transpose(1, 2, 0) * 2.5, 0, 1)
    gt = mask.numpy()
    pred = (prob > 0.5).astype(np.float32)

    # Extract boundaries
    gt_boundary = gt - ndimage.binary_erosion(gt).astype(np.float32)
    pred_boundary = pred - ndimage.binary_erosion(pred).astype(np.float32)

    # Overlay boundaries on RGB
    display = rgb.copy()
    display[gt_boundary > 0] = [0, 1, 0]  # green = GT boundary
    display[pred_boundary > 0] = [1, 0, 0]  # red = pred boundary

    ax.imshow(display)
    m = compute_metrics(logits.cpu(), mask.unsqueeze(0))
    ax.set_title(f"IoU={m['iou']:.2f}", fontsize=10)
    ax.axis("off")

# Legend
legend_patches = [
    mpatches.Patch(color="green", label="Ground Truth Boundary"),
    mpatches.Patch(color="red", label="Predicted Boundary"),
]
fig.legend(handles=legend_patches, loc="lower center", ncol=2, fontsize=11)
plt.suptitle("Burn Boundary Comparison (Green=GT, Red=Pred)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("boundary_comparison.png", dpi=150, bbox_inches="tight")
plt.show()